# ЛР 8. Скрининг с журналом и контролями

**Сущность:** Развести качество ассая, структурный риск и ранжирование; не удалять спорные записи молча.

**Данные:** molecules.csv плюс заданные учебные контроли μ=100/10, σ=5/5 и библиотека N=1000,A=20,k=50,a=5.

**Workflow:** Структуры → предупреждения → статусы → Z′ → EF → следующий контроль.

Перед запуском прочитайте [guideline](../guides/lab08.md) и [теорию](../../Module_theory/Module%208_Screening.md).

## 1. Прогноз

Запишите ожидаемое поведение и отрицательный контроль. Какое изменение входа должно изменить результат, а какое — сохранить?

In [1]:
from pathlib import Path
import sys, json, math, csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# WHY: relative discovery makes the notebook portable across machines.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Labs" / "chemistry.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите notebook внутри репозитория курса")
sys.path.insert(0, str(ROOT / "Labs"))
DATA = ROOT / "Labs" / "datasets"
OUT = ROOT / "Labs" / "results" / "lab08"
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()


## 2. Разбор и исполнение

1. Получите все PAINS-предупреждения, не объявляя механизм доказанным.
2. Посчитайте Z′ и EF на отдельных числах.
3. Запишите следующий контроль для review и объяснение для eligible.

Ожидаемое: Катехол review, пустой ввод invalid; Z′=2/3; EF@50=5; журнал 12 строк.

In [2]:
from chemistry import screen
from metrics import z_prime,enrichment
raw=pd.read_csv(DATA/"molecules.csv",keep_default_na=False)
records=[{"id":r.id,**screen(r.smiles)} for r in raw.itertuples()]
table=pd.DataFrame(records);display(table[["id","status","reason"]])
assert screen("Oc1ccccc1O")["status"]=="review"
assert screen("")["status"]=="invalid"
assert math.isclose(z_prime(100,10,5,5),2/3)
assert enrichment(1000,20,50,5)==5
table.to_csv(OUT/"decisions.csv",index=False)
artifact={"Z_prime":z_prime(100,10,5,5),"EF_at_50":enrichment(1000,20,50,5),"statuses":table.status.value_counts().to_dict()}


[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Normalizer
[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Normalizer
[11:58:47] Running LargestFragmentChooser
[11:58:47] Running Uncharger
[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Normalizer
[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Normalizer
[11:58:47] Running LargestFragmentChooser
[11:58:47] Running Uncharger
[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Normalizer
[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Norma

[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Normalizer
[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Normalizer
[11:58:47] Running LargestFragmentChooser
[11:58:47] Running Uncharger


,id,status,reason
0,M01,eligible,No listed alert; activity remains unknown
1,M02,eligible,No listed alert; activity remains unknown
2,M03,eligible,No listed alert; activity remains unknown
3,M04,eligible,No listed alert; activity remains unknown
4,M05,eligible,No listed alert; activity remains unknown
5,M06,review,PAINS:catechol_A(92)
6,M07,eligible,No listed alert; activity remains unknown
7,M08,invalid,Invalid SMILES
8,M09,invalid,SMILES must be a nonempty string
9,M10,eligible,No listed alert; activity remains unknown


[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Normalizer
[11:58:47] Initializing MetalDisconnector
[11:58:47] Running MetalDisconnector
[11:58:47] Initializing Normalizer
[11:58:47] Running Normalizer
[11:58:47] Running LargestFragmentChooser
[11:58:47] Running Uncharger


## 3. Проверяемая запись запуска

Benchmark фиксирует этот запуск. Время включает расчёт и создание артефактов, но не установку среды; это не сравнительный рейтинг оборудования.

In [3]:
elapsed = time.perf_counter()-started
run_record = {"lab": 8, "calculation_wall_seconds": elapsed, "result": artifact}
(OUT/"benchmark.json").write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding="utf-8")
print("Запуск сохранён:",OUT/"benchmark.json")


Запуск сохранён: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\practice-audit-issue1-20261117\Labs\results\lab08\benchmark.json


## 4. Самостоятельная тренировка

1. Увеличьте σ до 20 и объясните знак Z′.
2. Почему EF нельзя вычислить, если истинное число активных неизвестно?
3. Рассчитайте условные затраты контроля с учётом цены ложного отказа.

[Ответы после попытки](../solutions/lab08.md). Вернитесь к первому заданию через неделю.

## 5. Ваш научный вывод

1. Какой результат получен?
2. Какой вывод из него пока не следует?
3. Какой следующий эксперимент или контроль нужен?

Сохраните свой вариант notebook отдельно; исходный пример должен остаться воспроизводимым.